# Figure 6 revised: four representative event weeks — fixed input version

Reviewer #2 requested several weekly maps so that temporal variation of the risk surface can be assessed.
This version fixes the input-file path search and the national display extent: it uses the existing standardized prediction table that the executed `20_v4` visualization notebook actually selected, rather than assuming a nonexistent `14_v7_all_grid_predictions_event_weeks.parquet`.

Selection rule (to avoid cherry-picking): sort matched strict event weeks chronologically and use four approximately equally spaced weeks. The same 0–1 within-week percentile scale is used in all panels.

This v3 additionally requires an exact grid_id + week_start match before a week can be selected, so every panel necessarily contains at least one observed strict-event marker.


In [ ]:
# ============================================================
# 0. Google Drive mount and load the SAME prediction source
#    that was actually used by 20_v4 / the current Figure 6
# ============================================================
!pip -q install geopandas pyogrio

import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped/failed:', repr(e))

BASE_DIR = Path('/content/drive/MyDrive/avian_influenza_project')
PROC_DIR = BASE_DIR / 'processed'
MODEL_DIR = PROC_DIR / 'model_outputs_riskmap_eval'
OUT_DIR = MODEL_DIR / 'figure6_multiweek_reviewer2'
OUT_DIR.mkdir(parents=True, exist_ok=True)

XLIM = (123.5, 149.0)  # full 5,491-grid study extent; manuscript lon range 123.79–148.85
YLIM = (24.0, 46.0)   # full 5,491-grid study extent; manuscript lat range 24.25–45.52
MANUAL_WEEKS = None  # 例: ['2025-11-03','2025-12-15','2026-01-26','2026-03-23']

# IMPORTANT:
# The previous draft incorrectly assumed that
# 14_v7_all_grid_predictions_event_weeks.parquet existed.
# In the actually executed 20_v4 notebook, the selected source was:
#   .../20_v3_visualization_outputs_japan_heatmap/
#   20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv
# Therefore this notebook uses that existing standardized prediction table first,
# while retaining recursive fallbacks for later versions.
MYDRIVE = Path('/content/drive/MyDrive')

PRED_CANDIDATES = [
    # Locations used by different versions of the visualization workflow.
    MODEL_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv',
    PROC_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv',
    BASE_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv',
    MYDRIVE / '20_v3_visualization_outputs_japan_heatmap' / '20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv',

    MODEL_DIR / '20_v4_riskmap_style_japan_heatmap_outputs' / '20_v4_standardized_week_grid_risk_predictions_for_heatmap.csv',
    PROC_DIR / '20_v4_riskmap_style_japan_heatmap_outputs' / '20_v4_standardized_week_grid_risk_predictions_for_heatmap.csv',

    # Historical final-v7 all-grid prediction file, if present.
    MODEL_DIR / '14_v7_all_grid_predictions_event_weeks.parquet',
    PROC_DIR / '14_v7_all_grid_predictions_event_weeks.parquet',
]

# The actually executed 20_v4 notebook selected the following 49-event case file.
EVENT_CANDIDATES = [
    MODEL_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v2_standardized_19v2_gain_loss.csv',
    PROC_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v2_standardized_19v2_gain_loss.csv',
    BASE_DIR / '20_v3_visualization_outputs_japan_heatmap' / '20_v2_standardized_19v2_gain_loss.csv',
    MYDRIVE / '20_v3_visualization_outputs_japan_heatmap' / '20_v2_standardized_19v2_gain_loss.csv',

    MODEL_DIR / '14_v7_strict_first_occurrence_events_used.csv',
    PROC_DIR / '14_v7_strict_first_occurrence_events_used.csv',
    MODEL_DIR / '14_v5_strict_first_occurrence_events_used.csv',
    MODEL_DIR / '10_event_occurrence_type_classification.csv',
    PROC_DIR / '10_event_occurrence_type_classification.csv',
    MODEL_DIR / '13_best_model_first_occurrence_cases_with_success_label.csv',
]

def first_existing(paths):
    return next((Path(p) for p in paths if Path(p).exists()), None)

def pick_col(columns, exact=(), contains=()):
    cols = list(columns)
    low = {c: str(c).lower() for c in cols}
    for name in exact:
        for c in cols:
            if low[c] == str(name).lower():
                return c
    for key in contains:
        for c in cols:
            if str(key).lower() in low[c]:
                return c
    return None

def read_table(path):
    path = Path(path)
    if path.suffix.lower() == '.parquet':
        return pd.read_parquet(path)
    return pd.read_csv(path)

# ---------- prediction file ----------
pred_path = first_existing(PRED_CANDIDATES)

if pred_path is None:
    # Fallback 1: search only inside the avian_influenza_project tree.
    # This is much safer than assuming one particular subfolder.
    hits = sorted(BASE_DIR.rglob('20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv'))
    if hits:
        pred_path = hits[-1]

if pred_path is None:
    hits = sorted(BASE_DIR.rglob('*standardized_week_grid_risk_predictions_for_heatmap*.csv'))
    if hits:
        pred_path = hits[-1]

if pred_path is None:
    # Fallback 2: the executed visualization output folder may have been created
    # directly under MyDrive rather than inside avian_influenza_project.
    direct_folder = MYDRIVE / '20_v3_visualization_outputs_japan_heatmap'
    if direct_folder.exists():
        hits = sorted(direct_folder.glob('*standardized_week_grid_risk_predictions_for_heatmap*.csv'))
        if hits:
            pred_path = hits[-1]

if pred_path is None:
    print('Checked prediction candidates:')
    for p in PRED_CANDIDATES:
        print(' -', p, 'exists=', p.exists())
    raise FileNotFoundError(
        '20_v3_standardized_week_grid_risk_predictions_for_heatmap.csv が見つかりません。\n'
        'このファイルは現在のFigure 6作成に使われた週×5,491グリッド予測表です。\n'
        'Google Drive内の 20_v3_visualization_outputs_japan_heatmap フォルダの位置を確認してください。'
    )

raw = read_table(pred_path)
print('Prediction source:', pred_path)
print('Prediction shape:', raw.shape)
print('Prediction columns:', raw.columns.tolist())

# Standardize the columns used below. This supports both the existing 20_v3 CSV
# and a possible later 14_v7 parquet.
grid_col = pick_col(raw.columns, exact=['grid_id','mesh_id','cell_id'])
week_col = pick_col(raw.columns, exact=['week_start','event_week','target_week','week'])
lon_col = pick_col(raw.columns, exact=['grid_lon','centroid_lon','lon','longitude'])
lat_col = pick_col(raw.columns, exact=['grid_lat','centroid_lat','lat','latitude'])
pct_col = pick_col(raw.columns, exact=['risk_percentile','risk_percentile_within_week','percentile'], contains=['percentile'])
rank_col = pick_col(raw.columns, exact=['risk_rank','risk_rank_within_week','rank'], contains=['rank'])
top_col = pick_col(raw.columns, exact=['top10_hit','top10_flag','is_top10'], contains=['top10'])
model_col = pick_col(raw.columns, exact=['model_name','model'])

required_detected = {
    'grid': grid_col, 'week': week_col, 'lon': lon_col,
    'lat': lat_col, 'percentile': pct_col
}
missing = [k for k,v in required_detected.items() if v is None]
if missing:
    raise ValueError(f'Prediction fileで必要列を判定できません: {missing}. Columns={raw.columns.tolist()}')

pred = pd.DataFrame({
    'grid_id': raw[grid_col].astype(str),
    'week_start': pd.to_datetime(raw[week_col], errors='coerce'),
    'grid_lon': pd.to_numeric(raw[lon_col], errors='coerce'),
    'grid_lat': pd.to_numeric(raw[lat_col], errors='coerce'),
    'risk_percentile': pd.to_numeric(raw[pct_col], errors='coerce'),
})

if rank_col is not None:
    pred['risk_rank'] = pd.to_numeric(raw[rank_col], errors='coerce')
else:
    # Rank 1 = highest risk. Average rank handles ties.
    pred['risk_rank'] = pred.groupby('week_start')['risk_percentile'].rank(method='average', ascending=False)

if top_col is not None:
    x = raw[top_col]
    if pd.api.types.is_bool_dtype(x):
        pred['top10_hit'] = x.to_numpy()
    else:
        pred['top10_hit'] = x.astype(str).str.lower().isin(['true','1','yes','y']).to_numpy()
else:
    pred['top10_hit'] = pred['risk_rank'] <= np.ceil(pred.groupby('week_start')['grid_id'].transform('nunique') * 0.10)

# If a multi-model file is ever supplied, keep the final external GIS model when identifiable.
if model_col is not None:
    raw_models = raw[model_col].astype(str)
    preferred = raw_models.str.lower().str.contains('external') & raw_models.str.lower().str.contains('gis')
    if preferred.any():
        keep_index = raw.index[preferred]
        pred = pred.loc[keep_index].copy()
        print('Filtered model rows:', raw_models.loc[keep_index].value_counts().head().to_dict())

pred = pred.dropna(subset=['week_start','grid_lon','grid_lat','risk_percentile']).copy()
pred = pred.drop_duplicates(['grid_id','week_start'], keep='last')

# ---------- strict first-occurrence-like event file ----------
event_path = first_existing(EVENT_CANDIDATES)
if event_path is None:
    # Prefer the exact event/gain-loss file stored with the 20_v3 visualization products.
    hits = sorted(BASE_DIR.rglob('20_v2_standardized_19v2_gain_loss.csv'))
    if hits:
        event_path = hits[-1]

if event_path is None:
    direct_folder = MYDRIVE / '20_v3_visualization_outputs_japan_heatmap'
    if direct_folder.exists():
        hits = sorted(direct_folder.glob('20_v2_standardized_19v2_gain_loss.csv'))
        if hits:
            event_path = hits[-1]

if event_path is None:
    # Last-resort project search.
    hits = sorted(BASE_DIR.rglob('*gain*loss*.csv'))
    if hits:
        event_path = hits[-1]

if event_path is None:
    raise FileNotFoundError(
        'strict first-occurrence-like event file が見つかりません。'
    )

events_raw = read_table(event_path)
print('Event source:', event_path)
print('Event shape:', events_raw.shape)
print('Event columns:', events_raw.columns.tolist())

ev_grid_col = pick_col(events_raw.columns, exact=['grid_id','mesh_id','cell_id'])
ev_week_col = pick_col(events_raw.columns, exact=['week_start','event_week','target_week','week'])
if ev_grid_col is None or ev_week_col is None:
    raise ValueError(f'Event file must contain grid/week columns: {events_raw.columns.tolist()}')

events = events_raw.copy()
# If this is a full occurrence-classification table, restrict to the primary 8-week/30-km strict definition.
if 'occurrence_type_8w_30km' in events.columns:
    events = events[events['occurrence_type_8w_30km'].astype(str) == 'first_occurrence_like'].copy()
elif 'occurrence_type' in events.columns:
    vals = events['occurrence_type'].astype(str).str.lower()
    strict_mask = vals.str.contains('first')
    if strict_mask.any():
        events = events[strict_mask].copy()

events = pd.DataFrame({
    'grid_id': events[ev_grid_col].astype(str).str.strip(),
    'week_start': pd.to_datetime(events[ev_week_col], errors='coerce'),
}).dropna().drop_duplicates()

# IMPORTANT: select only event weeks for which the *event grid itself* can be matched
# to the 5,491-grid prediction table. This prevents a selected panel from having
# no observable event marker because of a grid-ID mismatch.
pred['grid_id'] = pred['grid_id'].astype(str).str.strip()
pred_keys = pred[['grid_id','week_start']].drop_duplicates()
matched_events = events.merge(pred_keys, on=['grid_id','week_start'], how='inner')

if len(matched_events) < len(events):
    unmatched = events.merge(pred_keys, on=['grid_id','week_start'], how='left', indicator=True)
    unmatched = unmatched[unmatched['_merge'] == 'left_only'][['grid_id','week_start']]
    print('WARNING: strict event rows not matched to prediction grid-week:', len(unmatched))
    if len(unmatched):
        display(unmatched.head(20))

# Use only weeks with at least one matched strict event grid.
event_weeks = sorted(pd.to_datetime(matched_events['week_start'].dropna().unique()))

print('Prediction weeks:', pred['week_start'].min().date(), 'to', pred['week_start'].max().date(), 'n =', pred['week_start'].nunique())
print('Prediction grids:', pred['grid_id'].nunique())
print(
    'Prediction coordinate range:',
    f"lon={pred['grid_lon'].min():.2f}–{pred['grid_lon'].max():.2f}, "
    f"lat={pred['grid_lat'].min():.2f}–{pred['grid_lat'].max():.2f}"
)
if not (
    pred['grid_lon'].between(XLIM[0], XLIM[1], inclusive='both').all()
    and pred['grid_lat'].between(YLIM[0], YLIM[1], inclusive='both').all()
):
    raise ValueError('At least one of the 5,491 grid centroids lies outside XLIM/YLIM.')
print('Matched strict event weeks:', len(event_weeks))

if len(event_weeks) < 4:
    raise ValueError(f'4週以上のmatched strict event weeksが必要ですが、{len(event_weeks)}週しかありません。')

if MANUAL_WEEKS is None:
    # Objectively spaced across the chronologically available strict event weeks.
    idx = np.round(np.linspace(0, len(event_weeks)-1, 4)).astype(int)
    selected_weeks = [event_weeks[i] for i in idx]
else:
    selected_weeks = [pd.Timestamp(x) for x in MANUAL_WEEKS]
    missing_weeks = [w for w in selected_weeks if w not in event_weeks]
    if missing_weeks:
        raise ValueError(f'MANUAL_WEEKS に予測＋strict eventのない週があります: {missing_weeks}')

print('Selected weeks:', [w.strftime('%Y-%m-%d') for w in selected_weeks])


In [ ]:
# ============================================================
# 1. Draw 2 x 2 multi-week Figure 6
# ============================================================
# Natural Earth boundary layers
countries_url = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip'
admin1_url = 'https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_1_states_provinces.zip'
world = gpd.read_file(countries_url)
japan = world[world['ADMIN'] == 'Japan'].copy()
admin1 = gpd.read_file(admin1_url)
if 'adm0_name' in admin1.columns:
    japan_admin1 = admin1[admin1['adm0_name'] == 'Japan'].copy()
elif 'admin' in admin1.columns:
    japan_admin1 = admin1[admin1['admin'] == 'Japan'].copy()
else:
    japan_admin1 = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

fig, axes = plt.subplots(2, 2, figsize=(12.2, 11.0), sharex=True, sharey=True)
axes = axes.ravel()
scatter_for_cbar = None
caption_rows = []

for ax, week in zip(axes, selected_weeks):
    week_df = pred[pred['week_start'] == week].copy()
    n_grids = week_df['grid_id'].nunique()
    if n_grids != 5491:
        raise ValueError(f'{week.date()}: expected 5,491 grids, found {n_grids}')

    top10_df = week_df[week_df['top10_hit'].astype(bool)].copy()
    event_ids = matched_events.loc[matched_events['week_start'] == week, 'grid_id'].astype(str).tolist()
    event_df = week_df[week_df['grid_id'].astype(str).isin(event_ids)].copy()
    if event_df.empty:
        raise ValueError(f'{week.date()}: selected event week has no matched event grid; check grid IDs.')

    japan.plot(ax=ax, color='white', edgecolor='black', linewidth=0.7, zorder=1)
    if len(japan_admin1) > 0:
        japan_admin1.boundary.plot(ax=ax, color='lightgray', linewidth=0.30, zorder=2)

    sc = ax.scatter(
        week_df['grid_lon'], week_df['grid_lat'], c=week_df['risk_percentile'],
        cmap='YlOrRd', vmin=0, vmax=1, marker='s', s=15, linewidths=0,
        alpha=0.95, zorder=3, rasterized=True
    )
    scatter_for_cbar = sc

    ax.scatter(
        top10_df['grid_lon'], top10_df['grid_lat'],
        facecolors='none', edgecolors='black', marker='s',
        s=28, linewidths=0.30, alpha=0.80, zorder=4, rasterized=True
    )

    if not event_df.empty:
        ax.scatter(
            event_df['grid_lon'], event_df['grid_lat'],
            facecolors='white', edgecolors='black', marker='o',
            s=105, linewidths=1.3, zorder=5
        )
        ax.scatter(
            event_df['grid_lon'], event_df['grid_lat'],
            color='black', marker='x', s=115, linewidths=2.2, zorder=6
        )

    ax.set_title(week.strftime('%d %B %Y'), fontsize=12)
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, linewidth=0.25, alpha=0.25)
    panel_i = list(selected_weeks).index(week)
    ax.set_xlabel('Longitude' if panel_i >= 2 else '')
    ax.set_ylabel('Latitude' if panel_i % 2 == 0 else '')

    for _, r in event_df.iterrows():
        caption_rows.append({
            'week_start': week.strftime('%Y-%m-%d'),
            'grid_id': str(r['grid_id']),
            'risk_percentile': float(r['risk_percentile']),
            'risk_rank': float(r['risk_rank']),
            'top10': bool(r['risk_rank'] <= np.ceil(5491 * 0.10)),
        })

# Shared legend and colorbar
top10_handle = Line2D([0],[0], marker='s', linestyle='None', markerfacecolor='none', markeredgecolor='black', markersize=7)
event_circle = Line2D([0],[0], marker='o', linestyle='None', markerfacecolor='white', markeredgecolor='black', markersize=9)
event_x = Line2D([0],[0], marker='x', linestyle='None', color='black', markersize=8, markeredgewidth=2.0)
axes[0].legend(
    [top10_handle, (event_circle, event_x)],
    ['Top 10% risk grid cells', 'Observed strict event grid'],
    handler_map={tuple: HandlerTuple(ndivide=1)},
    loc='upper left', frameon=True, fontsize=8
)

fig.subplots_adjust(right=0.90, wspace=0.08, hspace=0.12)
cax = fig.add_axes([0.92, 0.17, 0.018, 0.66])
cbar = fig.colorbar(scatter_for_cbar, cax=cax)
cbar.set_label('Risk percentile within the week')
cbar.set_ticks([0,0.2,0.4,0.6,0.8,1.0])

out_png = OUT_DIR / 'Figure6_external_GIS_riskmap_four_representative_event_weeks.png'
out_pdf = OUT_DIR / 'Figure6_external_GIS_riskmap_four_representative_event_weeks.pdf'
fig.savefig(out_png, dpi=300, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
plt.show()

caption_df = pd.DataFrame(caption_rows)
selected_df = pd.DataFrame({'selected_week':[w.strftime('%Y-%m-%d') for w in selected_weeks]})
caption_df.to_csv(OUT_DIR / 'Figure6_four_week_event_values.csv', index=False, encoding='utf-8-sig')
selected_df.to_csv(OUT_DIR / 'Figure6_four_week_selection.csv', index=False, encoding='utf-8-sig')

print('Saved:', out_png)
print('Saved:', out_pdf)
print('Top10 cells by selected week:')
print(pred[pred['week_start'].isin(selected_weeks)].groupby('week_start')['top10_hit'].sum())
display(caption_df)


## Suggested caption

**Figure 6.** Example weekly national HPAI risk maps for four chronologically spaced strict first-occurrence-like event weeks with exact event-grid matches in the weekly prediction dataset. All 5,491 grid cells are colored by their within-week risk percentile using a common 0–1 scale. Grid cells in the highest-risk decile are outlined, and observed strict first-occurrence-like event grids are indicated by circled × markers. The weeks were selected by an objective chronological spacing rule rather than by model performance, allowing visual assessment of how the spatial prioritization surface changes over time. Map lines delineate the study area and do not necessarily depict accepted national boundaries.
